# Chat with the LLMs

This notebook sets up a chat interface with a chosen model (base or finetuned) and provides the output with and without RAG.

## Prerequisites

The RAG implementation requires the vector database is pre-generated. The GitHub Actions workflow should keep it updated.

If not, run the [document generation script](retrieval_db_update.ipynb).

Using the fine-tuned model requires the chat model has been trained on the data.

If it hasn't been run yet, run the [model finetuning notebook](finetuning.ipynb).

In [1]:
# Chat LLM model name
CHAT_MODEL = "Qwen/Qwen2.5-14B-Instruct"
# Fine-tuned model directory
FT_CHAT_MODEL = f"/opt/shared/data/models/{CHAT_MODEL}-Finetuned"

In [2]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
import ipywidgets as widgets

from retrieval import Retrieval

In [3]:
# Check GPU availability
print(f"CUDA Available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0)}")

CUDA Available: True
GPU: NVIDIA A100 80GB PCIe


In [4]:
# Take the model name without the organization part
# e.g. Qwen/Qwen2.5-14B-Instruct -> Qwen2.5-14B-Instruct
chat_model_name = CHAT_MODEL.split('/')[1]

# Name of the options
sel_base = chat_model_name
sel_ft = f'{chat_model_name}-Finetuned'

# Create a selection widget to pick a model 
model_select = widgets.ToggleButtons(
    value=sel_base,
    options=[sel_base, sel_ft],
    description='',
    disabled=False,
    button_style='',
    style={'button_width': '250px'}
)

model_select

ToggleButtons(options=('Qwen2.5-14B-Instruct', 'Qwen2.5-14B-Instruct-Finetuned'), style=ToggleButtonsStyle(but…

In [5]:
# Use default embeddings model, sentence-transformers/all-distilroberta-v1
# Use the default ChromaDB directory and collection
retrieval = Retrieval(log=True)

Loading the sentence transformer, sentence-transformers/all-distilroberta-v1 ...
Loading ChromaDB, /opt/shared/data/chromadb ...
Loading collection, all-documents ...
Initializing retrieval done


In [8]:
# Load the model and tokenizer

# convert from the selection name back to the model name that the `transformers` library would understand
modelname = ""
if model_select.value == sel_base:
    modelname = CHAT_MODEL
elif model_select.value == sel_ft:
    modelname = FT_CHAT_MODEL

print(f"Loading model and tokenizer: {modelname}")

chat_tokenizer = AutoTokenizer.from_pretrained(
    modelname,
    device_map="auto",
)
# Ensure padding token is set
if chat_tokenizer.pad_token is None:
    chat_tokenizer.pad_token = tokenizer.eos_token

print("Tokenizer loaded")

chat_model = AutoModelForCausalLM.from_pretrained(
    modelname,
    dtype=torch.bfloat16,
    trust_remote_code=True,
    device_map="auto"
)

print("Model loaded")

# Create the pipeline
pipe = pipeline(
    "text-generation",
    model=chat_model,
    tokenizer=chat_tokenizer,
    device_map="auto"
)

Loading model and tokenizer: /opt/shared/data/models/Qwen/Qwen2.5-14B-Instruct-Finetuned
Tokenizer loaded


Loading checkpoint shards:   0%|          | 0/6 [00:00<?, ?it/s]

Device set to use cuda:0


Model loaded


In [9]:
import copy
# adds some nice features to `input`
import readline

# Maintain conversation history
conversation = [
    {"role": "system", "content": "You are a helpful assistant."},
]

# Maintain RAG conversation history
rag_conversation = copy.deepcopy(conversation)
rag_conversation[0]["content"] += "\nDocuments will be included with the user's queries according to a naive RAG system."
rag_conversation[0]["content"] += "\nIgnore documents that do not contain any relevant information whatsoever. Adhere to them when relevant."
rag_conversation[0]["content"] += "\nDo not mention the documents/context or refer to them directly. Repeat key information from them where pertinent."

# Reset the RAG document injection history
retrieval.reset()

while True:
    user_input = input("> ")

    if not user_input or user_input in ["quit", "q", "exit"]:
        break

    for rag, history in [(False, conversation), (True, rag_conversation)]:
        if rag:
            user_input = retrieval.augment(user_input, 3, log=True)
        
        # Add user message to conversation
        history.append({"role": "user", "content": user_input})

        # Pass the conversation directly to the pipeline
        outputs = pipe(
            history,
            max_new_tokens=1024,
            # To ensure repeatability, always pick the most-likely candidate for the next token.
            # To achieve this, turn sampling off (~= setting temperature to zero, but torch doesn't like that)
            do_sample=False,
            temperature=None,
            top_p=None,
            top_k=None
        )
    
        # Extract the assistant's response
        response = outputs[0]["generated_text"][-1]["content"]
    
        # Add assistant response to conversation
        history.append({"role": "assistant", "content": response})
        
        print()
        if rag:
            print("RAG+", end="")
        print("LLM>", response)
        print()

>  What is CHPC?



LLM> CHPC stands for the Centre for High Performance Computing. It typically refers to the national facility in South Africa that provides high performance computing services, resources and expertise to the research, science and technology community in South Africa. However, the term can also refer to similar organizations in other countries or contexts. If you're referring to a specific CHPC, could you provide more details so I can give a more accurate answer?

[RAG] INJECTING (0.5625163912773132): readme-structure-of-the-competition-1.md
[RAG] INJECTING (0.5987644195556641): readme-1.md
[RAG] IRRELEVANT (0.6282262802124023): NOT INJECTING FILE: tutorial3_readme-high-performance-conjugate-gradients-1.md

RAG+LLM> The CHPC stands for the Center for High Performance Computing. It invites applications from qualified candidates to participate in the CHPC Student Cluster Competition, which aims to give undergraduate students at South African universities exposure to the HPC industry.



>  q
